# Sámi Language AI — Research Starter Notebook

**Goal:** quantify the real data gap across the Sámi languages, pull in the datasets that actually exist and are open, and set up the skeleton for an evaluation benchmark.

**Run this on Kaggle (New Notebook → enable Internet in settings) or Google Colab.** It will not run inside a sandboxed offline environment because it needs to reach `huggingface.co` and `zenodo.org`.

Datasets used here are real and verified as of Sept 2026:
- `tartuNLP/smugri4-data` — token counts across all Sámi languages
- `ltg/saami-web` — Northern Sámi Web Corpus (UiO Language Technology Group)
- Zenodo record 14176353 — Northern Sami YleAreena Subtitle Corpus

Not included here because they are *not* openly downloadable (noted for the governance chapter, not a bug): LIA Sápmi, Giellagas spoken Sámi corpus — both require a data-use agreement with the maintainers.

## 0. Setup

In [ ]:
!pip install -q datasets huggingface_hub pandas requests fasttext

import pandas as pd
from datasets import load_dataset
import requests
import json

## 1. Quantify the gap: token counts across all Sámi languages

`tartuNLP/smugri4-data` is the single best place to see the resource imbalance in one table. This is the number you'll cite in your proposal's motivation section.

In [ ]:
# The repo mixes many Uralic languages with incompatible per-file schemas (e.g. Estonian
# dialect subsets add extra keys), which breaks datasets' automatic Arrow merge.
# Fix: list only the Sami-language files directly and parse the raw JSON ourselves.

from huggingface_hub import list_repo_files, hf_hub_download
import json
from collections import defaultdict

sami_iso_codes = ["sjd", "sju", "sma", "sme", "smj", "smn"]  # Kildin, Ume, South, North, Lule, Inari

all_files = list_repo_files("tartuNLP/smugri4-data", repo_type="dataset")
sami_files = [f for f in all_files if f.split("/")[0] in sami_iso_codes and f.endswith(".json")]

print(f"Found {len(sami_files)} Sami-language files across {len(sami_iso_codes)} languages")
for code in sami_iso_codes:
    count = len([f for f in sami_files if f.startswith(code + "/")])
    print(f"  {code}: {count} files")

In [ ]:
# Download and parse each file individually (robust to schema differences between files),
# accumulating real token counts and a text sample per language.

token_totals = defaultdict(int)
doc_counts = defaultdict(int)
samples = defaultdict(list)

for fpath in sami_files:
    lang = fpath.split("/")[0]
    local_path = hf_hub_download("tartuNLP/smugri4-data", fpath, repo_type="dataset")
    try:
        with open(local_path, encoding="utf-8") as f:
            data = json.load(f)
    except json.JSONDecodeError:
        continue  # skip malformed files rather than crashing the whole pass

    entries = data if isinstance(data, list) else [data]
    for entry in entries:
        features = entry.get("features", {})
        counter = features.get("counter_tokens", {})
        for k, v in counter.items():
            base_lang = k.split("_")[0]  # normalize e.g. 'est_hiiu' -> 'est'
            if base_lang in sami_iso_codes:
                token_totals[base_lang] += v
        doc_counts[lang] += 1
        # grab one short text sample per language for a sanity check later
        if len(samples[lang]) < 1:
            texts = entry.get("texts") or entry.get("sentences") or []
            if texts:
                samples[lang].append(str(texts[0])[:200])

print("Real token counts recovered directly from source files:")
for lang in sami_iso_codes:
    print(f"  {lang}: {token_totals[lang]:,} tokens across {doc_counts[lang]} files")

In [ ]:
iso_to_name = {
    "sjd": "Kildin Sami", "sju": "Ume Sami", "sma": "Southern Sami",
    "sme": "Northern Sami", "smj": "Lule Sami", "smn": "Inari Sami",
}

gap_df = pd.DataFrame([
    {"language": iso_to_name[k], "tokens": token_totals[k]} for k in sami_iso_codes
])
gap_df["pct_of_largest"] = (gap_df["tokens"] / gap_df["tokens"].max() * 100).round(3)
gap_df = gap_df.sort_values("tokens", ascending=False).reset_index(drop=True)
gap_df

**Note:** Skolt Sámi and Pite Sámi don't even appear in this dataset — that absence *is* your data point. Worth stating explicitly in the proposal: even the most comprehensive open multi-Sámi resource currently available has no Skolt or Pite Sámi text at all.

### 1a. Visualize and interpret the imbalance

Log scale is necessary here — on a linear scale, five of the six languages would look like zero.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(gap_df["language"], gap_df["tokens"])
ax.set_xscale("log")
ax.set_xlabel("Tokens (log scale)")
ax.set_title("Open text resources across Sámi languages (tartuNLP/smugri4-data)")
plt.tight_layout()
plt.savefig("sami_resource_gap.png", dpi=150)
plt.show()

print(f"Northern Sami has {gap_df['tokens'].max() / gap_df.loc[gap_df['language']=='Kildin Sami','tokens'].values[0]:,.0f}x more tokens than Kildin Sami.")
print("Skolt Sami and Pite Sami: 0 tokens — absent from this dataset entirely.")

**For the proposal, this paragraph is now defensible, not asserted:**

> Even the most comprehensive open multi-Sámi text resource currently available (tartuNLP's smugri4-data) shows a greater than 1,700x gap in token count between the best- and worst-resourced living Sámi languages with any data at all, and contains zero tokens of Skolt or Pite Sámi — despite both being actively spoken, documented languages with published linguistic research. This is not a hypothetical gap; it is measurable in the field's own most-used open resource.

## 2. Load the Northern Sámi Web Corpus (ltg/saami-web)

Built by UiO's Language Technology Group via web crawl seeded from Sámi Wikipedia. CC0-licensed, 33k rows.

In [ ]:
saami_web = load_dataset("ltg/saami-web", split="train")
df_web = saami_web.to_pandas()

print(f"Rows: {len(df_web)}")
print(df_web.columns.tolist())
df_web.head(3)

In [ ]:
# Basic text stats — sentence/word length distribution, useful for designing benchmark item difficulty later
if "text" in df_web.columns:
    df_web["word_count"] = df_web["text"].str.split().str.len()
    print(df_web["word_count"].describe())

## 3. Pull the Northern Sami YleAreena Subtitle Corpus (Zenodo)

Small, clean, broadcast-sourced — good for a manually-checkable pilot sample rather than large-scale training.

In [ ]:
# Zenodo record 14176353 — file: sami_subtitles.json
zenodo_record_id = "14176353"
r = requests.get(f"https://zenodo.org/api/records/{zenodo_record_id}")
record = r.json()

# Find the download link for sami_subtitles.json
files = record.get("files", [])
for f in files:
    print(f["key"], "->", f["links"]["self"])

In [ ]:
# Download and load the subtitle corpus
subtitle_url = [f["links"]["self"] for f in files if f["key"] == "sami_subtitles.json"][0]
subs = requests.get(subtitle_url).json()

print(f"Type: {type(subs)}")
print(json.dumps(subs[:3] if isinstance(subs, list) else list(subs.items())[:3], indent=2, ensure_ascii=False))

## 4. Skeleton: morphological evaluation stub

This is where your actual contribution starts. Sámi languages are heavily agglutinative — a real benchmark needs to test whether a model produces correct inflected forms, not just plausible-looking text. Below is a minimal, honest skeleton — it does **not** contain a working morphological analyzer; that has to come from GiellaLT's FST tools (see note below), which are not a `pip install`, they're compiled from source.

**Next real step (not in this notebook):** clone the GiellaLT Skolt Sámi FST repo from GitHub (`giellalt` organization) and build it locally — that's the actual morphological ground truth Rueter's group built. This notebook only gets you the raw text/token side; the FST tooling needs its own setup, which is worth doing as a second session once you've confirmed with Rueter that this is a sound direction.

In [ ]:
def stub_morphology_check(word: str, language: str) -> dict:
    """
    Placeholder only. Real evaluation requires the GiellaLT FST analyzer
    for the target language (built from github.com/giellalt), not a Python library.
    This function documents the intended interface for later integration.
    """
    return {
        "word": word,
        "language": language,
        "status": "NOT IMPLEMENTED — requires compiled GiellaLT FST analyzer",
    }

stub_morphology_check("sámegiella", "sme")

## 4a. Handle the long-document outliers — split, don't discard

Correction from earlier: manual inspection shows the longest documents are official/institutional sources — Sámi allaskuvla, Sametinget, Giellatekno, a Sámi hymnal, and what appear to be university theses from a repository (samas.brage.unit.no). That's valuable formal-register text, not scraped junk. Split long documents into paragraph-sized chunks instead of dropping them, so the corpus keeps this material without one row dominating any length-based statistics.

In [ ]:
outliers = df_web[df_web["word_count"] > df_web["word_count"].quantile(0.99)]
print(f"{len(outliers)} long documents (99th percentile+) — inspecting sources before deciding what to do:")
print(outliers[["url", "word_count"]].sort_values("word_count", ascending=False).head(10))

def split_into_chunks(text: str, max_words: int = 500) -> list:
    words = text.split()
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

long_doc_mask = df_web["word_count"] > df_web["word_count"].quantile(0.99)
short_docs = df_web[~long_doc_mask].copy()

chunked_rows = []
for _, row in df_web[long_doc_mask].iterrows():
    for i, chunk in enumerate(split_into_chunks(row["text"])):
        chunked_rows.append({"text": chunk, "url": row["url"] + f"#chunk{i}"})

df_web_clean = pd.concat([short_docs, pd.DataFrame(chunked_rows)], ignore_index=True)
print(f"\nOriginal: {len(df_web)} rows -> after splitting long docs: {len(df_web_clean)} rows")
print("No text discarded — the formal/institutional material is preserved, just chunked.")

## 4b. Language identification pass on the Zenodo subtitle corpus

The file is labelled "Northern Sami Subtitle Corpus," but manual inspection shows mixed orthography — plausibly North Sámi, Inari Sámi, and plain Finnish in the same file with no per-sentence language label. Run GlotLID (the same identifier used to build `ltg/saami-web`) to check properly rather than relying on the file's own label.

In [ ]:
!pip install -q fasttext huggingface_hub

from huggingface_hub import hf_hub_download
import fasttext
import numpy as np

# GlotLID: open-source language identification covering 2000+ languages including Sami variants
model_path = hf_hub_download(repo_id="cis-lmu/glotlid", filename="model.bin")
lid_model = fasttext.load_model(model_path)

# Monkeypatch: fasttext's own predict() calls np.array(probs, copy=False), which numpy>=2.0
# rejects outright (this is a fasttext packaging bug, not fixable by us upstream, and pip-pinning
# numpy mid-session doesn't help since numpy is already imported). Swap that one call for
# np.asarray, which is safe on every numpy version, and leave the rest of the method identical.
def _patched_predict(self, text, k=1, threshold=0.0, on_unicode_error='strict'):
    def check(entry):
        if entry.find('\n') != -1:
            raise ValueError("predict processes one line at a time (remove 'n')")
        entry += "\n"
        return entry
    text = check(text)
    predictions = self.f.predict(text, k, threshold, on_unicode_error)
    if len(predictions):
        probs, labels = zip(*predictions)
    else:
        probs, labels = ([], ())
    return labels, np.asarray(probs)

fasttext.FastText._FastText.predict = _patched_predict

def identify_lang(text: str) -> str:
    text = text.replace("\n", " ").strip()
    if not text:
        return "empty"
    label, conf = lid_model.predict(text, k=1)
    return label[0].replace("__label__", "")  # e.g. 'sme_Latn', 'smn_Latn', 'fin_Latn'

In [ ]:
# Flatten the sentence list from the subtitle JSON pulled in section 3, then tag each sentence
sentences_dict = dict(subs)["sentences"] if isinstance(subs, list) else subs["sentences"]
sub_df = pd.DataFrame(sentences_dict)
sub_df["detected_lang"] = sub_df["text"].apply(identify_lang)

lang_counts = sub_df["detected_lang"].value_counts()
print("Detected language breakdown in the 'Northern Sami Subtitle Corpus':")
print(lang_counts)
print(f"\n{(lang_counts.get('sme_Latn', 0) / len(sub_df) * 100):.1f}% actually confirmed as North Sami (sme).")

**If this confirms the mix (likely sme + smn + fin, possibly others):** that's a genuine, citable finding for your proposal — the field's own labeled resources are not always what their label says, which is exactly the kind of quality-control gap a rigorous evaluation benchmark needs to catch. Worth mentioning directly to Rueter and in your governance framing: who verifies these labels, and how, matters as much as building new data.

## 4c. Verify the short-utterance-noise hypothesis

Before claiming the 16 rare detected languages are classifier noise rather than real content, check it: if the hypothesis is right, those sentences should be dramatically shorter (in tokens) than the confidently-classified ones. This is the kind of check a reviewer will ask for, so do it now rather than assert it.

In [ ]:
main_langs = {"sme_Latn", "smn_Latn", "fin_Latn"}
sub_df["is_rare_lang"] = ~sub_df["detected_lang"].isin(main_langs)

print("Token-count comparison: main languages vs. rare/noise-flagged detections")
print(sub_df.groupby("is_rare_lang")["tokens"].describe())

# Show the actual rare-language sentences for manual inspection
print("\nAll sentences flagged as rare languages (for manual spot-check):")
print(sub_df[sub_df["is_rare_lang"]][["text", "tokens", "detected_lang"]].to_string(index=False))

**Interpretation:** if the rare-language group's mean/median token count is near 1–2 and the main-language group's is much higher, that confirms the noise hypothesis — GlotLID (like most LID models) needs several tokens of context and is unreliable on single-word utterances ("Hei!", "Antti.", "Joo."). Report this check explicitly in the paper's methodology section rather than just asserting it.

## 4d. Save final numbers for the paper (reproducibility artifact)

Write every verified number from this session to one JSON file. This is what you cite from and what you can attach as supplementary material — a reviewer or Rueter/Bongo can re-run this exact notebook and get the same numbers back.

In [ ]:
import datetime

final_results = {
    "generated": datetime.datetime.now().isoformat(),
    "token_counts_by_language": {
        iso_to_name[k]: int(token_totals[k]) for k in sami_iso_codes
    },
    "file_counts_by_language": {
        iso_to_name[k]: int(doc_counts[k]) for k in sami_iso_codes
    },
    "north_to_kildin_ratio": round(token_totals["sme"] / token_totals["sjd"], 1),
    "languages_absent_from_smugri4": ["Skolt Sami (sms)", "Pite Sami (sje)"],
    "saami_web_corpus": {
        "source": "ltg/saami-web (Hugging Face)",
        "total_rows_original": int(len(df_web)),
        "total_rows_after_chunking": int(len(df_web_clean)),
        "long_doc_sources_examples": outliers["url"].head(5).tolist(),
    },
    "subtitle_corpus_language_audit": {
        "source": "Zenodo record 14176353 (Northern Sami Subtitle Corpus)",
        "total_sentences": int(len(sub_df)),
        "language_breakdown": sub_df["detected_lang"].value_counts().to_dict(),
        "pct_confirmed_north_sami": round(lang_counts.get("sme_Latn", 0) / len(sub_df) * 100, 1),
        "rare_lang_token_stats": sub_df.groupby("is_rare_lang")["tokens"].describe().to_dict(),
    },
}

with open("sami_nlp_audit_results.json", "w", encoding="utf-8") as f:
    json.dump(final_results, f, indent=2, ensure_ascii=False)

print("Saved sami_nlp_audit_results.json")
print(json.dumps(final_results, indent=2, ensure_ascii=False)[:1500], "...")

## 4e. Pilot benchmark: can a general-purpose LLM distinguish Sámi varieties?

This is the first real evaluation experiment, not just a data audit. Question: given a sentence confidently labelled by GlotLID as North Sámi, Inari Sámi, or Finnish, can a general-purpose LLM correctly identify which it is? If models can't reliably tell closely related Sámi varieties apart, that's direct evidence for why a dedicated Sámi evaluation benchmark is needed — which is exactly the proposal's thesis, now with a first experiment behind it.

**Requires an API key.** On Kaggle: Add-ons → Secrets → add `ANTHROPIC_API_KEY`, then attach it to this notebook.

In [ ]:
!pip install -q anthropic

from kaggle_secrets import UserSecretsClient
import anthropic

api_key = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=api_key)

MODEL = "claude-sonnet-4-5"  # swap for whichever model you have access to

In [ ]:
# Build a balanced gold-standard pilot set from sub_df (already computed in 4b).
# Cap at the smallest class size (Finnish, n=14) so the test isn't trivially
# dominated by the majority class — a balanced test is the honest version of this.

label_map = {"sme_Latn": "North Sami", "smn_Latn": "Inari Sami", "fin_Latn": "Finnish"}
eval_pool = sub_df[sub_df["detected_lang"].isin(label_map.keys())].copy()
eval_pool["gold_label"] = eval_pool["detected_lang"].map(label_map)

class_counts = eval_pool["gold_label"].value_counts()
n_per_class = class_counts.min()
print(f"Class sizes before balancing: {class_counts.to_dict()}")
print(f"Balancing to {n_per_class} per class (n={n_per_class*3} total pilot set)")

pilot_set = (
    eval_pool.groupby("gold_label", group_keys=False)
    .apply(lambda g: g.sample(n=n_per_class, random_state=42))
    .reset_index(drop=True)
)
pilot_set[["text", "gold_label"]]

In [ ]:
import time

PROMPT_TEMPLATE = (
    "Which language is the following sentence written in? "
    "Answer with exactly one of these three options and nothing else: "
    "'North Sami', 'Inari Sami', 'Finnish'.\n\nSentence: {text}"
)

def classify_sentence(text: str) -> str:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(text=text)}],
    )
    return resp.content[0].text.strip()

predictions = []
for _, row in pilot_set.iterrows():
    pred = classify_sentence(row["text"])
    predictions.append(pred)
    time.sleep(0.3)  # gentle rate limiting for a ~42-call pilot

pilot_set["model_prediction"] = predictions
pilot_set["correct"] = pilot_set["model_prediction"] == pilot_set["gold_label"]
pilot_set[["text", "gold_label", "model_prediction", "correct"]]

In [ ]:
overall_accuracy = pilot_set["correct"].mean()
print(f"Overall accuracy: {overall_accuracy:.1%} (n={len(pilot_set)})\n")

print("Per-class accuracy:")
print(pilot_set.groupby("gold_label")["correct"].mean())

print("\nConfusion matrix (rows=gold, columns=predicted):")
confusion = pd.crosstab(pilot_set["gold_label"], pilot_set["model_prediction"])
print(confusion)

# Save for the paper
pilot_results = {
    "model": MODEL,
    "n_total": len(pilot_set),
    "n_per_class": int(n_per_class),
    "overall_accuracy": round(float(overall_accuracy), 4),
    "per_class_accuracy": pilot_set.groupby("gold_label")["correct"].mean().round(4).to_dict(),
    "confusion_matrix": confusion.to_dict(),
}
with open("sami_lid_pilot_results.json", "w", encoding="utf-8") as f:
    json.dump(pilot_results, f, indent=2, ensure_ascii=False)
print("\nSaved sami_lid_pilot_results.json")

## 5. Summary table for the proposal

Run this last to produce the one table worth pasting into your gap-mapping section.

In [ ]:
summary = pd.DataFrame([
    {"resource": "tartuNLP/smugri4-data", "covers": "6 languages, token counts", "open": True, "source": "Hugging Face"},
    {"resource": "ltg/saami-web", "covers": "North Sami only", "open": True, "source": "Hugging Face"},
    {"resource": "YleAreena Subtitle Corpus", "covers": "North Sami only, small", "open": True, "source": "Zenodo"},
    {"resource": "LIA Sápmi speech", "covers": "North Sami dialects, spoken", "open": False, "source": "UiO — agreement required"},
    {"resource": "Giellagas spoken corpus", "covers": "North Sami spoken", "open": False, "source": "Oulu — agreement required"},
    {"resource": "GiellaLT Skolt Sami FST", "covers": "Skolt Sami morphology, 30k+ words", "open": True, "source": "GitHub (compile required)"},
])
summary